In [4]:
!pip install -q cerebras_cloud_sdk openai instructor pydantic chromadb reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.1/106.1 kB 6.3 MB/s eta 0:00:00


## schema and injestion

In [12]:
import os
import re
import json
from datetime import datetime, timedelta
from typing import List, Optional
import pandas as pd
import matplotlib.pyplot as plt
import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel, Field
from cerebras.cloud.sdk import Cerebras
import instructor
from google.colab import userdata, files

# PDF Generation imports
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, HRFlowable

from openai import OpenAI
import instructor
from google.colab import userdata

# 1. Initialize standard OpenAI client configured for Cerebras Cloud
cerebras_api_key = userdata.get('CEREBRAS_API_KEY')

raw_cerebras_client = OpenAI(
    base_url="https://api.cerebras.ai/v1",
    api_key=cerebras_api_key
)

# 2. Patch using instructor.from_openai
instructor_client = instructor.from_openai(
    raw_cerebras_client,
    mode=instructor.Mode.JSON
)

# Plain text client for Cell 5
plain_text_client = raw_cerebras_client

# 2. HBI Pydantic Schema
class ExtractedSymptoms(BaseModel):
    general_wellbeing: int = Field(..., ge=0, le=4, description="0=Very well, 1=Slightly below par, 2=Poor, 3=Very poor, 4=Terrible")
    abdominal_pain: int = Field(..., ge=0, le=3, description="0=None, 1=Mild, 2=Moderate, 3=Severe")
    liquid_stool_count: int = Field(..., ge=0, description="Count of loose or liquid bowel movements (Bristol 6 or 7 only). Solid stools = 0.")
    bristol_types: List[int] = Field(default_factory=list, description="Reported Bristol types (1-7).")
    complications: List[str] = Field(default_factory=list, description="Extraintestinal symptoms: arthralgia, mouth ulcers, etc.")
    medication_adherence: Optional[bool] = Field(None, description="True if meds confirmed taken, False if missed, None if unmentioned.")

def extract_daily_symptoms(raw_note: str) -> ExtractedSymptoms:
    return instructor_client.chat.completions.create(
        model="qwen-3.8-27b",  # Fast and reliable for JSON schema extraction on Cerebras
        response_model=ExtractedSymptoms,
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract daily Crohn's symptoms strictly adhering to the Harvey-Bradshaw Index. "
                    "Only count liquid or watery stools (Bristol 6-7) toward liquid_stool_count. Formed stools count as 0."
                )
            },
            {"role": "user", "content": raw_note}
        ],
        temperature=0.0
    )

## Data Ingestion, Deterministic Analytics & Temporal RAG



In [18]:
import os
import re
import json
import time
from datetime import datetime, timedelta
from typing import List, Optional
import pandas as pd
import matplotlib.pyplot as plt
import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel, Field
from openai import OpenAI, RateLimitError, APIError
import instructor
from google.colab import userdata, files

# PDF Generation imports
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, HRFlowable

# 1. Cerebras Client Initialization
cerebras_api_key = userdata.get('CEREBRAS_API_KEY')
raw_cerebras_client = OpenAI(
    base_url="https://api.cerebras.ai/v1",
    api_key=cerebras_api_key
)

instructor_client = instructor.from_openai(
    raw_cerebras_client,
    mode=instructor.Mode.JSON
)
plain_text_client = raw_cerebras_client

# 2. Schema
class DailySymptomRecord(BaseModel):
    day: int
    general_wellbeing: int = Field(..., ge=0, le=4, description="0=Very well, 1=Slightly below par, 2=Poor, 3=Very poor, 4=Terrible")
    abdominal_pain: int = Field(..., ge=0, le=3, description="0=None, 1=Mild, 2=Moderate, 3=Severe")
    liquid_stool_count: int = Field(..., ge=0, description="Count of loose or liquid bowel movements (Bristol 6 or 7 only). Solid stools = 0.")
    complications: List[str] = Field(default_factory=list, description="Extraintestinal symptoms: arthralgia, mouth ulcers, etc.")
    medication_adherence: Optional[bool] = Field(None, description="True if meds confirmed taken, False if missed, None if unmentioned.")

class BatchSymptoms(BaseModel):
    records: List[DailySymptomRecord]

# 3. Extraction with Built-in Backoff for High-Traffic Queues
def extract_all_symptoms_batch(logs: List[str], max_attempts: int = 5) -> List[DailySymptomRecord]:
    formatted_input = "\n".join([f"Day {i}: {note}" for i, note in enumerate(logs, 1)])
    delay = 2.0

    for attempt in range(1, max_attempts + 1):
        try:
            res = instructor_client.chat.completions.create(
                model="qwen-3.8-27b",  # or "llama3.1-8b"
                response_model=BatchSymptoms,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Extract daily Crohn's symptoms for each day according to the Harvey-Bradshaw Index. "
                            "Only count liquid or watery stools (Bristol 6-7). Solid stools = 0."
                        )
                    },
                    {"role": "user", "content": formatted_input}
                ],
                temperature=0.0
            )
            return res.records
        except (RateLimitError, APIError, Exception) as e:
            if attempt == max_attempts:
                raise e
            print(f"Cerebras traffic spike ({e}). Retrying in {delay}s (attempt {attempt}/{max_attempts})...")
            time.sleep(delay)
            delay *= 2.0

# 4. 14 Days of Patient Logs (Input)
base_date = datetime.now() - timedelta(days=14)
raw_logs = [
    # Baseline (Days 1-5)
    "Felt good. Normal formed stool once. Ate chicken and rice.",
    "No stomach pain. Normal solid bowel movement. Went to the gym.",
    "Felt slightly below par. Bowel was normal. Took my morning azathioprine.",
    "Normal day. Energy levels good. 1 solid stool. Ate some spicy Thai curry.",
    "Felt fine, slight bloating in evening after dinner. Stool was Bristol 4.",
    # Onset (Days 6-9)
    "Woke up with mild cramps. Stool was mushy twice. Skipped evening medication by mistake.",
    "Mild pain continued, 2 loose watery stools. Feeling worn out.",
    "Knee joints started aching badly. Stool loose 3 times. Felt fatigued.",
    "Had a small mouth ulcer appear. 3 watery stools, moderate lower right quadrant pain.",
    # Flare Peak (Days 10-14)
    "Struggling with fatigue. Moderate cramping. 4 liquid stools. Left knee swollen.",
    "Severe abdominal cramping around 3pm. 4 watery stools. Poor energy.",
    "Terrible day. Exhausted. 5 watery diarrhea episodes. Severe cramping.",
    "Knee still painful, mouth ulcer hurting. 4 loose stools, severe pain. Took pain relief.",
    "Severe cramps before every bowel movement. 5 liquid stools. Felt terrible all day."
]

# 5. Extract Metrics and Build History using batch extraction
extracted_records = extract_all_symptoms_batch(raw_logs)

processed_history = []
for i, ext in enumerate(extracted_records, start=1):
    hbi = ext.general_wellbeing + ext.abdominal_pain + ext.liquid_stool_count + len(ext.complications)
    processed_history.append({
        "day": i,
        "date": (base_date + timedelta(days=i-1)).strftime("%Y-%m-%d"),
        "raw": raw_logs[i-1],
        "hbi": hbi,
        "pain": ext.abdominal_pain,
        "liquid": ext.liquid_stool_count,
        "comps": ext.complications,
        "meds": ext.medication_adherence if ext.medication_adherence is not None else True
    })

# 6. Deterministic Analytics (Zero LLM Math)
total_days = len(processed_history)
baseline_hbi = sum(d['hbi'] for d in processed_history if 1 <= d['day'] <= 5) / 5
current_hbi = sum(d['hbi'] for d in processed_history if 10 <= d['day'] <= 14) / 5
missed_days = [d['day'] for d in processed_history if not d['meds']]

# Ensure unique complications and clean up names (e.g., 'mouth ulcer' and 'mouth ulcers' become 'Mouth Ulcer')
cleaned_comps = set()
for d in processed_history:
    for comp in d['comps']:
        # Simple normalization: lowercase and remove 's' at the end for plural
        normalized_comp = comp.lower().replace(' ulcers', ' ulcer')
        normalized_comp = normalized_comp.replace('left knee swollen', 'knee swelling')
        cleaned_comps.add(normalized_comp.capitalize())

all_comps = list(cleaned_comps)

# Format adherence detail for better readability
adherence_detail_str = "Full adherence recorded"
if missed_days:
    if len(missed_days) == 1:
        adherence_detail_str = f"Missed evening dose on Day {missed_days[0]}"
    else:
        adherence_detail_str = f"Missed evening dose on Days {', '.join(map(str, missed_days))}"

stats_summary = {
    "monitoring_window_days": total_days,
    "baseline_hbi_avg": round(baseline_hbi, 1),
    "current_hbi_avg": round(current_hbi, 1),
    "hbi_trend_delta": round(current_hbi - baseline_hbi, 1),
    "medication_adherence_percent": round((sum(1 for d in processed_history if d['meds'])
                                          / total_days) * 100, 1),
    "missed_medication_days": missed_days,
    "adherence_detail": adherence_detail_str,
    "total_liquid_stools_reported": sum(d['liquid'] for d in processed_history),
    "reported_complications": all_comps
}

# 7. Ingest into In-Memory Chroma Vector DB
chroma_client = chromadb.Client()
emb_fn = embedding_functions.DefaultEmbeddingFunction()
log_collection = chroma_client.create_collection(name="crohns_logs", embedding_function=emb_fn, get_or_create=True)

for entry in processed_history:
    log_collection.add(
        documents=[entry["raw"]],
        metadatas=[{"day": entry["day"], "date": entry["date"], "hbi": entry["hbi"], "meds": entry["meds"]}],
        ids=[f"day_{entry['day']}"]
    )

# 8. Retrieve Context with Temporal Anchors
def retrieve_anchored_context(query_str: str) -> str:
    res = log_collection.query(query_texts=[query_str], n_results=3, include=["documents", "metadatas"])
    paired = sorted(zip(res["documents"][0], res["metadatas"][0]), key=lambda x: x[1]["day"])
    return "\n".join([f"- [Day {m['day']} | HBI: {m['hbi']} | Meds: {m['meds']}]: \"{d}\"" for d, m in paired])

formatted_triggers = retrieve_anchored_context("dietary foods eaten spicy curry chicken meals")
formatted_complications = retrieve_anchored_context("joint pain knee aching mouth ulcers swelling")

print("Processing complete. Key Stats:", stats_summary)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 91.6MiB/s]


Processing complete. Key Stats: {'monitoring_window_days': 14, 'baseline_hbi_avg': 0.2, 'current_hbi_avg': 10.8, 'hbi_trend_delta': 10.6, 'medication_adherence_percent': 92.9, 'missed_medication_days': [6], 'adherence_detail': 'Missed evening dose on Day 6', 'total_liquid_stools_reported': 32, 'reported_complications': ['Arthralgia', 'Mouth ulcer', 'Knee swelling']}


## Chart & PDF Report Engine

In [19]:
def build_hbi_chart(history: list, output_path: str = "trajectory.png"):
    df = pd.DataFrame(history)
    df['hbi_rolling'] = df['hbi'].rolling(window=3, min_periods=1, center=True).mean()

    plt.figure(figsize=(9.5, 3.2), dpi=200)
    ax = plt.subplot(111)

    # Standard Clinical Severity Bands
    ax.axhspan(0, 4.9, color='#e8f5e9', alpha=0.75, label="Remission (<5)")
    ax.axhspan(5, 7.9, color='#fff9c4', alpha=0.75, label="Mild (5-7)")
    ax.axhspan(8, max(df['hbi'].max() + 2, 14), color='#ffebee', alpha=0.75, label="Moderate/Severe (8+)")

    # Dual Layer: Raw Points + Rolling Trend
    ax.plot(df['day'], df['hbi'], marker='o', markersize=3.5, color='#90caf9', linestyle=':', alpha=0.6, label="Daily HBI")
    ax.plot(df['day'], df['hbi_rolling'], color='#1565c0', linewidth=2.2, label="3-Day Rolling Trend")

    ax.set_title("Longitudinal Harvey-Bradshaw Index (HBI) Trajectory", fontsize=9, fontweight='bold', pad=8)
    ax.set_xlabel("Monitoring Day", fontsize=8)
    ax.set_ylabel("HBI Score", fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_xticks(df['day'])
    ax.tick_params(labelsize=7)
    ax.legend(loc="upper left", framealpha=0.9, fontsize=7)

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.savefig(output_path, bbox_inches='tight')
    plt.close()

def clean_text_for_pdf(text: str) -> str:
    """Sanitizes unicode spaces, hyphens, and unescaped ampersands to avoid ReportLab tofu blocks."""
    if not text:
        return ""
    text = re.sub(r'[\u2010\u2011\u2012\u2013\u2014\u2015\u2212]', '-', text)
    text = re.sub(r'[\u00A0\u2000-\u200B\u202F\u205F\u3000]', ' ', text)
    text = re.sub(r'&(?!(?:amp|lt|gt|quot|apos|#\d+|#x[a-fA-F0-9]+);)', '&amp;', text)
    text = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', text)
    return text

def compile_clinical_pdf(stats: dict, summary_text: str, chart_path: str, pdf_filename: str):
    doc = SimpleDocTemplate(pdf_filename, pagesize=A4, leftMargin=36, rightMargin=36, topMargin=32, bottomMargin=32)
    styles = getSampleStyleSheet()

    header_style = ParagraphStyle('HeaderTitle', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=13, leading=15, textColor=colors.HexColor('#0d47a1'))
    sub_style = ParagraphStyle('SubHeader', parent=styles['Normal'], fontName='Helvetica', fontSize=7.5, leading=9, textColor=colors.HexColor('#555555'))
    section_style = ParagraphStyle('SectionHeading', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=9, leading=11, textColor=colors.HexColor('#0d47a1'), spaceBefore=5, spaceAfter=2)
    body_style = ParagraphStyle('BodyTextCustom', parent=styles['Normal'], fontName='Helvetica', fontSize=7.5, leading=9.5, textColor=colors.HexColor('#222222'))
    table_text = ParagraphStyle('TText', parent=styles['Normal'], fontName='Helvetica', fontSize=7, leading=9)
    table_header = ParagraphStyle('THead', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=7, leading=9, textColor=colors.HexColor('#0d47a1'))

    story = [
        Paragraph("CLINICAL OUTPATIENT CONSULTATION BRIEF", header_style),
        Paragraph(f"Generated: {datetime.now().strftime('%d %b %Y')} | Assessment: Harvey-Bradshaw Index (HBI) & BSFS | Non-Diagnostic Decision Support", sub_style),
        Spacer(1, 4),
        HRFlowable(width="100%", thickness=1.2, color=colors.HexColor('#0d47a1'), spaceAfter=6)
    ]

    comps_display = ", ".join(stats['reported_complications']) if stats['reported_complications'] else "None reported"
    table_data = [
        [
            Paragraph("<b>Window:</b>", table_header), Paragraph(f"{stats['monitoring_window_days']} Days", table_text),
            Paragraph("<b>Baseline HBI:</b>", table_header), Paragraph(str(stats['baseline_hbi_avg']), table_text),
            Paragraph("<b>Current HBI:</b>", table_header), Paragraph(str(stats['current_hbi_avg']), table_text)
        ],
        [
            Paragraph("<b>HBI Trend Δ:</b>", table_header), Paragraph(f"+{stats['hbi_trend_delta']}" if stats['hbi_trend_delta'] > 0 else str(stats['hbi_trend_delta']), table_text),
            Paragraph("<b>Med Adherence:</b>", table_header), Paragraph(f"{stats['medication_adherence_percent']}%", table_text),
            Paragraph("<b>Total Liquid Stools:</b>", table_header), Paragraph(str(stats['total_liquid_stools_reported']), table_text)
        ],
        [
            Paragraph("<b>Complications:</b>", table_header), Paragraph(clean_text_for_pdf(comps_display.title()), table_text),
            Paragraph("<b>Clinical Tier:</b>", table_header), Paragraph("Moderate Flare" if stats['current_hbi_avg'] >= 8 else "Mild Activity", table_text),
            Paragraph("<b>Adherence Note:</b>", table_header), Paragraph(stats['adherence_detail'], table_text)
        ]
    ]

    t = Table(table_data, colWidths=[75, 95, 75, 95, 85, 95])
    t.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#f5f7fa')),
        ('BOX', (0,0), (-1,-1), 0.5, colors.HexColor('#cfd8dc')),
        ('INNERGRID', (0,0), (-1,-1), 0.3, colors.HexColor('#eceff1')),
        ('TOPPADDING', (0,0), (-1,-1), 2),
        ('BOTTOMPADDING', (0,0), (-1,-1), 2),
    ]))
    story.append(t)
    story.append(Spacer(1, 4))

    # Parse and Sanitize Summary Paragraphs
    clean_lines = re.sub(r'#+\s*', '', summary_text).strip().split('\n')
    for line in clean_lines:
        line = clean_text_for_pdf(line.strip())
        if not line:
            continue
        if any(line.startswith(prefix) for prefix in ["1.", "2.", "3.", "4.", "Baseline", "Onset", "Escalation", "Overall"]):
            story.append(Paragraph(f"<b>{line}</b>", section_style))
        elif line.startswith(("-", "*")):
            story.append(Paragraph(f"• {line.lstrip('-* ')}", body_style))
        else:
            story.append(Paragraph(line, body_style))

    story.append(Spacer(1, 4))

    # Inject Trajectory Chart
    if os.path.exists(chart_path):
        story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor('#b0bec5'), spaceAfter=4))
        story.append(Image(chart_path, width=520, height=170))

    doc.build(story)
    return pdf_filename

## Synthesis Generation & Automatic Download


In [21]:
# 1. Plot Trajectory Chart
chart_img_path = "hbi_trajectory.png"
build_hbi_chart(processed_history, chart_img_path)

# 2. Strict Phase-Based Prompt
SYNTHESIS_PROMPT = """You are an expert Clinical Data Synthesizer assisting an outpatient gastroenterologist.
Compile a 1-page Clinical Outpatient Brief based on 14 days of patient logs.
A visual HBI trajectory chart is appended at the bottom. Do not output day-by-day tables.

STRICT CONSTRAINTS:
1. ACCURATE MEDICATION SCOPE:
   - The global adherence percentage belongs ONLY in top-level summaries.
   - Do NOT cite the 14-day adherence percentage inside individual phase sections.
   - Baseline (Days 1–5): Note full adherence (100%).
   - Onset (Days 6–9): Explicitly report that an evening dose was missed on Day 6.
   - Escalation (Days 10–14): Note adherence was maintained.
2. 3-PHASE BREAKDOWN:
   - Baseline (Days 1–5): Remission status, baseline habits.
   - Onset (Days 6–9): Flare emergence, stool consistency shift, systemic signs, missed dose.
   - Escalation (Days 10–14): Peak severity, stool frequency, functional impact.
3. STRICT TRIGGER CORRELATION: Apply the 24–48h window before evaluating dietary triggers.
4. NON-PRESCRIPTIVE BOUNDS: Never recommend drugs, diets, or referrals. Summarize objective data.
Word limit: Under 160 words so it fits cleanly on one page with the chart."""

user_prompt_content = f"""DETERMINISTIC CLINICAL METRICS:
{json.dumps(stats_summary, indent=2)}

QUALITATIVE LOG CHUNKS:
Potential Dietary Triggers:
{formatted_triggers}

Complications & Symptoms:
{formatted_complications}"""

# 3. Call Llama 3.3 70B on Cerebras
response = plain_text_client.chat.completions.create(
    model="qwen-3.8-27b",
    messages=[
        {"role": "system", "content": SYNTHESIS_PROMPT},
        {"role": "user", "content": user_prompt_content}
    ],
    temperature=0.1,
    max_tokens=600
)

summary_markdown = response.choices[0].message.content or ""

print("\n=======================================================")
print("             GENERATED CLINICAL BRIEF")
print("=======================================================\n")
print(summary_markdown)

# 4. Generate PDF and Download
final_pdf_path = compile_clinical_pdf(
    stats=stats_summary,
    summary_text=summary_markdown,
    chart_path=chart_img_path,
    pdf_filename="Crohns_Clinical_Brief_Production.pdf"
)

print(f"\nReport compiled successfully: {final_pdf_path}")
files.download(final_pdf_path)


             GENERATED CLINICAL BRIEF



Report compiled successfully: Crohns_Clinical_Brief_Production.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>